# EDA #2 — `vehicles_clean.csv`

## Récapitulatif pour le rapport — comment on en est arrivé là

**Approche initiale (stations only)** : trois modèles XGBoost (`train_naive.py`, `train_xgboost.py`, `train_xgboost_weekly.py`) entraînés sur `stations_clean.csv` (lags + rollings + diffs + features cycliques + `current_value`). Résultats : **MAE 0.17-0.18 vs persistance 0.087** → tous les modèles **perdent contre une simple prédiction « rien ne change »**.

**Diagnostic** : 72 % d'importance sur `current_value` et 22 % sur `lag_1` ; toutes les autres features < 1 %. Le modèle est aveugle parce que toutes ses entrées dérivent du **même signal univarié `num_vehicles_available`** — ajouter un lag ou une moyenne mobile ne crée pas de nouvelle information.

**Pivot** : pour débloquer, il faut une source d'information **physique orthogonale** à la donnée stations. C'est le rôle de `vehicle_status.csv` (positions GPS, batterie, statut désactivé de chaque vélo). Ce notebook valide que cette source est exploitable et le préprocessing nécessaire avant fusion.

**Contenu** :
- §1 → §4 : EDA initiale (couverture, désactivés, batteries, transit).
- §5 : diagnostic d'une anomalie repérée en §3 (29 % de vélos à 0 % de batterie) → identification de **31 vélos fantômes** filtrés à la source.
- §6 : sanity check de jointure post-filtre → **GO pour la fusion**.
- §7 : conclusion + prochaines étapes (features spatiales).

**Constats préliminaires (issus du nettoyage `clean_vehicle_status.py`)** :
- `vehicle_type_id` constant à `x2` → droppé.
- `is_reserved` constant à `False` → droppé. `bikes_reserved` éliminée comme feature potentielle.
- ~49 % des lignes ont `station_index = 0` (vélo en circulation) — signal absent de `stations_clean`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

vehicles = pd.read_csv('../data/vehicles_clean.csv', parse_dates=['timestamp'])
stations = pd.read_csv('../data/stations_clean.csv', parse_dates=['timestamp'])
print(f'vehicles : {len(vehicles):,} lignes × {vehicles.shape[1]} cols')
print(f'stations : {len(stations):,} lignes × {stations.shape[1]} cols')
vehicles.head()

## §1 — Couverture temporelle

**Hypothèse** : la fenêtre temporelle couvre les mêmes ~26 jours que `stations_clean`. La taille de la flotte (`vehicle_id` uniques) devrait être stable, autour de 100 vélos.

In [ ]:
print(f"Plage vehicles : {vehicles['timestamp'].min()}  ->  {vehicles['timestamp'].max()}")
print(f"Plage stations : {stations['timestamp'].min()}  ->  {stations['timestamp'].max()}")
print(f"Vélos uniques  : {vehicles['vehicle_id'].nunique()}")

per_slot = vehicles.groupby('timestamp').size()
print(f"Vélos par snapshot : moy={per_slot.mean():.1f}  min={per_slot.min()}  max={per_slot.max()}")
per_slot.plot(figsize=(12, 3), title='Nombre de vélos rapportés par snapshot 30 min')
plt.show()

**Observation** : la flotte est légèrement plus grosse qu'attendu (≈ 97 vélos uniques après filtre fantômes — 128 avant filtre, voir §5). Le nombre de vélos rapportés par snapshot oscille autour de 90 ± quelques unités, sans rupture brutale → la donnée est continue.

**Conclusion** : la couverture temporelle est cohérente avec `stations_clean` après recadrage automatique fait au nettoyage. La taille de flotte > au max attendu indique qu'un signal supplémentaire — la **position individuelle de chaque vélo** — devient un atout : on connaît plus de vélos que ce que les compteurs station aggregués révèlent.

## §2 — Statut `is_disabled` (par station)

**Hypothèse** : la majorité des vélos sont actifs (`is_disabled=0`). Quelques stations concentrent les désactivés (vélos cassés en attente de maintenance).

In [ ]:
print(f"Taux global is_disabled : {vehicles['is_disabled'].mean():.1%}")

# Par station (hors transit)
in_station = vehicles[vehicles['station_index'] > 0]
by_station = in_station.groupby('station_index')['is_disabled'].agg(['sum', 'count', 'mean'])
by_station.columns = ['nb_disabled', 'nb_obs', 'taux_disabled']
print('\nTop 10 stations par taux de vélos désactivés :')
print(by_station.sort_values('taux_disabled', ascending=False).head(10).round(3).to_string())

**Observation** : taux global ≈ 14 %. Forte hétérogénéité entre stations : la station 4 monte à 24 %, la station 16 à 18 %, alors que d'autres sont sous 5 %. Ce sont probablement des stations « cimetière » où les vélos cassés s'accumulent.

**Conclusion** : `bikes_disabled` est un **signal exploitable et orthogonal à `num_vehicles_available`** (puisque la doc GBFS dit qu'`available` exclut déjà les désactivés). Cette feature mérite d'être ajoutée à la fusion future. Elle expliquerait pourquoi le modèle stations-only est parfois pessimiste : 5 vélos disponibles dont 2 désactivés ≠ 5 vélos prêts à partir.

## §3 — Distribution `current_fuel_percent`

**Hypothèse** : distribution bimodale (pleins ou vides). Une queue à gauche significative justifierait le seuil `< 0.20` pour la feature future `bikes_low_battery`.

In [ ]:
fuel = vehicles['current_fuel_percent'].dropna()
print(f"n = {len(fuel):,}")
print(f"moyenne = {fuel.mean():.2f}   médiane = {fuel.median():.2f}")
print(f"% < 0.20 (seuil low_battery) : {(fuel < 0.20).mean():.1%}")
print(f"% == 0.0  : {(fuel == 0.0).mean():.1%}")
print(f"% == 1.0  : {(fuel == 1.0).mean():.1%}")

fuel.hist(bins=50, figsize=(10, 3))
plt.title('Distribution de current_fuel_percent (après filtre fantômes)')
plt.xlabel('fuel %'); plt.ylabel('comptage')
plt.show()

**Observation INITIALE (avant filtre)** : 29.7 % des relevés à exactement 0 % de batterie ; 36 % sous 0.20 ; seulement 1.5 % à 100 %. Ce taux de « zéros » est anormalement élevé pour une flotte en exploitation.

**Hypothèse formée** : ces 0 % proviennent de vélos « fantômes » — des vélos perdus ou hors service que l'API continue de remonter, batterie figée à 0. À investiguer en §5 avant d'utiliser `current_fuel_percent` comme feature.

**Observation APRÈS filtre fantômes** (section §5) : la queue à gauche reste mais devient interprétable ; le seuil 0.20 est une approximation raisonnable de « vélo peu utile ».

## §4 — Vélos en circulation (`station_index = 0`)

**Hypothèse** : pic de vélos en transit en heure de pointe locale (7-9h, 17-19h). Corrélation négative attendue avec la somme `num_vehicles_available` du réseau.

In [ ]:
transit = vehicles[vehicles['station_index'] == 0].groupby('timestamp').size()
transit.name = 'in_transit'

# Profil horaire (heure UTC+4) — zoom Y pour voir les vraies oscillations
transit_df = transit.reset_index()
transit_df['hour'] = pd.to_datetime(transit_df['timestamp']).dt.hour
profile = transit_df.groupby('hour')['in_transit'].mean()
ax = profile.plot(kind='bar', figsize=(10, 3),
                  title='Vélos en circulation : moyenne par heure (UTC+4)')
ax.set_ylim(profile.min() * 0.97, profile.max() * 1.03)  # zoom Y pour amplifier les variations
ax.set_ylabel('vélos en transit (moy)')
plt.show()

# Corrélation avec dispo réseau
stations_total = stations.groupby('timestamp')['num_vehicles_available'].sum()
stations_total.index = pd.to_datetime(stations_total.index)
transit.index = pd.to_datetime(transit.index)
joined = pd.concat([transit, stations_total], axis=1, join='inner').dropna()
joined.columns = ['in_transit', 'available_total']
print(f"Corrélation transit ↔ dispo réseau : {joined.corr().iloc[0, 1]:.3f}")

**Observation** : corrélation ≈ **-0.60** entre vélos en transit et vélos disponibles agrégés sur le réseau. Très significative : quand la flotte est en circulation, les compteurs station baissent — physiquement attendu.

**Mais** : la variation absolue du nombre de vélos en transit est faible (oscillation entre ~50 et ~60 sur la fenêtre observée). Le profil horaire ne montre pas de pic franc 7h-9h ou 17h-19h ; les écarts sont visibles uniquement après zoom Y (l'axe complet écrase les variations).

**Conclusion** : le compte agrégé `nb_vélos_en_transit` est un signal globalement faible. **Mais** la corrélation -0.60 avec la dispo réseau valide qu'il y a bien de l'information physique. Le vrai gisement de signal se trouvera plutôt dans les **positions individuelles** des vélos en transit (proximité d'une station ⇒ arrivée imminente) — c'est ce qui justifie d'introduire les **features spatiales** comme prochaine étape (haversine vélo ↔ station).

## §5 — Diagnostic « 0 % batterie » : les vélos fantômes

Les 29.7 % de relevés à 0 % de batterie observés en §3 ont déclenché une investigation. Hypothèse : ces vélos sont des fantômes — des appareils perdus, cassés ou volés que l'API continue de remonter passivement, leur batterie figée à 0.

**Test** : un vélo en exploitation réelle est rechargé au moins une fois sur 26 jours. Donc un `vehicle_id` dont `max(current_fuel_percent) == 0` sur toute la fenêtre est un fantôme avec une très haute confiance.

Pour le démontrer, on recharge le CSV brut (avant filtre) et on identifie les coupables.

In [ ]:
raw = pd.read_csv('../data/vehicle_status.csv', parse_dates=['timestamp'])
max_fuel = raw.groupby('vehicle_id')['current_fuel_percent'].max()
phantoms = max_fuel[max_fuel == 0.0].index

print(f"Flotte totale (avant filtre)         : {raw['vehicle_id'].nunique()} vélos")
print(f"Fantômes détectés (jamais rechargés) : {len(phantoms)} vélos ({len(phantoms)/raw['vehicle_id'].nunique():.1%})")
print(f"Lignes représentées par les fantômes : {raw['vehicle_id'].isin(phantoms).sum():,} / {len(raw):,}")

# Vérification : les fantômes sont-ils marqués is_disabled ?
ph_rows = raw[raw['vehicle_id'].isin(phantoms)]
print(f"\nParmi les lignes fantômes : taux is_disabled = {ph_rows['is_disabled'].mean():.1%}")
print("-> l'API ne les marque PAS comme désactivés -> impossible à filtrer côté is_disabled seul.")

**Observation** : 31 vélos fantômes détectés sur 128, soit **24 % de la flotte apparente**. Ces vélos représentent 25 % des lignes du CSV brut. Ils ne sont **pas marqués `is_disabled`** par l'API — donc invisibles pour un filtre naïf. C'est pour ça qu'on a un critère temporel (« jamais rechargé ») plutôt qu'un critère de statut.

**Action prise** : `clean_vehicle_status.py` a été mis à jour avec `drop_phantom_vehicles()` qui élimine ces vélos à la source. Le CSV `vehicles_clean.csv` chargé en haut de ce notebook est déjà filtré.

**Conséquence** : la flotte « réelle » exploitable passe de 128 à 97 vélos, et la distribution de batterie devient bien plus interprétable.

## §6 — Sanity check de jointure (LE TEST DÉCISIF)

**But** : vérifier que pour un même `(timestamp, station_index)`, le nombre de vélos rattachés dans `vehicles_clean` (filtré des désactivés) est cohérent avec `num_vehicles_available` dans `stations_clean`.

**Pourquoi c'est critique** : si l'écart est systématique, alors toute fusion future sera basée sur du sable. C'est l'unique vérification qui valide notre **modèle de données**, pas seulement notre code.

**Hypothèse** : différence ≤ 1 sur la grande majorité des paires (drift toléré à cause de l'arrondi 30 min).

In [ ]:
# Agrégation : nombre de vélos actifs par (timestamp, station_index)
va = (vehicles[(vehicles['station_index'] > 0) & (vehicles['is_disabled'] == 0)]
      .groupby(['timestamp', 'station_index']).size().rename('vehicles_active'))
sa = stations.set_index(['timestamp', 'station_index'])['num_vehicles_available'].rename('num_avail')

cmp = pd.concat([va, sa], axis=1, join='inner').fillna(0)
cmp['diff'] = cmp['vehicles_active'] - cmp['num_avail']

print(f"paires (timestamp, station)   : {len(cmp):,}")
print(f"|diff| moyen                  : {cmp['diff'].abs().mean():.3f}")
print(f"% paires avec |diff| <= 1     : {(cmp['diff'].abs() <= 1).mean():.1%}")
print(f"% paires avec diff == 0 exact : {(cmp['diff'] == 0).mean():.1%}")
print("\nDistribution des écarts :")
print(cmp['diff'].astype(int).value_counts().sort_index().to_string())

# Top 5 stations avec écart systématique
by_st = cmp.groupby(level='station_index')['diff'].agg(['mean', 'count'])
by_st['abs_mean'] = by_st['mean'].abs()
print('\nTop 5 stations par écart absolu moyen :')
print(by_st.sort_values('abs_mean', ascending=False).head(5).round(3).to_string())

**Observation** : `|diff|` moyen ≈ **0.12** ; **98.5 % des paires ont `|diff| ≤ 1`** ; ≈ 89 % à diff = 0 exact. Aucune station n'a un écart systématique > 1 vélo en moyenne.

**Conclusion** : **GO pour la fusion.** Le mapping `station_id → station_index` est correct, l'arrondi 30 min n'introduit qu'un drift résiduel négligeable, et les sémantiques `is_disabled == 0` côté vélos vs `num_vehicles_available` côté stations coïncident.

**Note importante** : ce résultat n'a été atteint qu'**après** (i) le filtre fantômes (§5) et (ii) la régénération de `stations_clean.csv` pour combler les 5 h de retard sur la collecte. Avant ces deux corrections, le sanity check naïf donnait des écarts énormes — qui étaient des artefacts, pas des bugs sémantiques.

## §7 — Conclusion & prochaines étapes (rapport)

### Ce que ce notebook a établi

1. **`vehicle_status.csv` est exploitable** comme source de signal orthogonal à `stations_clean` (taux de vélos désactivés, batteries faibles, vélos en transit).
2. **Trois préprocessings non-triviaux** ont été nécessaires avant que la donnée soit utilisable :
   - drop des colonnes constantes (`vehicle_type_id`, `is_reserved`) ;
   - filtrage de **31 vélos fantômes** (24 % de la flotte) invisibles via le statut `is_disabled` seul ;
   - recadrage temporel sur la fenêtre commune avec `stations_clean` (résolu en régénérant côté stations).
3. **Sanity check de jointure validé** : agrégés par `(timestamp, station_index)`, les comptes vélos vs `num_vehicles_available` coïncident à ±1 vélo dans 98.5 % des cas. La fusion est techniquement saine.
4. **Enrichissement préparatoire de `stations_clean`** : ajout de `lat`, `lon`, `capacity` extraits de l'API GBFS `station_information.json`. Ces colonnes ne servent à rien telles quelles pour le modèle, mais débloquent les **features spatiales** (cf. ci-dessous).

### Ce qui justifie l'effort vis-à-vis de l'approche stations-only

Les modèles stations-only échouent car toutes leurs features dérivent du même signal univarié. Les features qu'on pourra extraire de `vehicles_clean` après fusion sont qualitativement nouvelles :
- `bikes_disabled` (varie de 0 à 24 % selon la station) ;
- `bikes_low_battery` (proxy de qualité réelle de l'offre) ;
- `mean_battery` (continu) ;
- **distance moyenne des vélos en transit à chaque station** (haversine, débloqué par `lat`/`lon` côté stations) — proxy d'**arrivées imminentes**, signal totalement absent du modèle stations-only.

### Prochaine itération

1. Création de `merge_vehicles_to_stations.py` produisant `stations_enriched.csv` avec :
   - les 4 features d'agrégation (`bikes_disabled`, `bikes_low_battery`, `mean_battery`, `bikes_in_radius_500m`) ;
   - haversine vélos en transit ↔ chaque station (rayon configurable).
2. Re-train de `train_xgboost.py` sur `stations_enriched.csv` — comparaison directe MAE/lift vs la run actuelle, en particulier sur les `HARD_STATIONS` (Port, Casabona, Eglise de Terre Sainte…).